## Deliverable 2
Markus Valland, Jørgen Skarra, Wiktor Correia-Myrhagen



In [63]:

import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

#Task 1 File ingestion and regular expressions
def read_file(path) -> tuple[str, str]:
    raw = Path(path).read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("iso-8859-1"), "iso-8859-1"

#Compile regex pattern for log line parsing 
LOG_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)

#Function to parse log lines into structured records
def parse_log(text: str) -> list[dict]:
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.match(line.strip().lstrip("\ufeff"))
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records

## Task 2: Recursive utility functions
### 2.1 Finding log files recursively



In [64]:
for item in Path("data").iterdir():
    print(item, "-> folder" if item.is_dir() else "-> file")

data/2026 -> folder
data/store_config.json -> file


In [65]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """finds all files with the given extension in a folder structure."""
    directory = Path(directory)
    log_files = []
    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item, extension))  # recursive call
        elif item.is_file() and item.suffix == extension:
            log_files.append(item)
    return log_files

In [66]:
log_files = find_log_files("data")
print(f"Found {len(log_files)} log files")
for f in log_files[:5]:
    print(f)

Found 9 log files
data/2026/science_building/POS_01/register_sales.log
data/2026/science_building/POS_03/register_sales.log
data/2026/science_building/POS_02/register_sales.log
data/2026/student_union/POS_01/register_sales.log
data/2026/student_union/POS_03/register_sales.log


### 2.2 Active registers from store_config.json
The function traverses any combination of nested dicts and lists:
- **dict**: if it has `"id"` and `"active": true`, the id is added. Then we search all values.
- **list**: we search every element.
- **anything else** (string, number, bool): base case, nothing to do.

In [67]:
def find_active_registers(data) -> set[str]:
    """Recursively traverses nested dicts and lists and returns the IDs of active registers."""
    active = set()
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active.add(data["id"])
        for value in data.values():
            active |= find_active_registers(value)   # recursive call
    elif isinstance(data, list):
        for element in data:
            active |= find_active_registers(element)  # recursive call
    return active

In [68]:
config = json.loads(Path("data/store_config.json").read_text(encoding="utf-8"))
active_registers = find_active_registers(config)
print(sorted(active_registers))

['POS_01', 'POS_02', 'POS_03']


## Task 3: Cleaning and analysing data using NumPy
### 3.1 Removing price outliers
A value is flagged as a hardware glitch if |x − μ| > 3σ. Outliers are replaced by the median of the valid (non-outlier) prices, so the glitch values do not influence the replacement value.

In [69]:
def remove_price_outliers(prices) -> np.ndarray:
    """Replaces values more than 3 standard deviations from the mean
    with the median of the valid prices. Returns a cleaned 1D array."""
    # Convert input to a 1D float array (np.array makes a copy, so the input is not modified)
    prices = np.array(prices, dtype=float).ravel()

    mean = prices.mean()
    std = prices.std()

    # Outliers deviate from the mean by more than 3 standard deviations
    is_outlier = np.abs(prices - mean) > 3 * std

    # The median is calculated from the valid prices only
    median = np.median(prices[~is_outlier])

    prices[is_outlier] = median
    return prices

In [70]:
test_prices = [4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 999.0]
cleaned = remove_price_outliers(test_prices)
print("Before:", test_prices)
print("After: ", cleaned)

Before: [4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 4.5, 3.0, 4.2, 3.5, 3.8, 7.5, 999.0]
After:  [4.5 3.  4.2 3.5 3.8 7.5 4.5 3.  4.2 3.5 3.8 7.5 4. ]


### 3.2 Refund ratio
Refund ratio = refund count / (sales count + ε), where ε = 10⁻⁶ prevents division by zero.

In [71]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Computes the element-wise refund ratio using NumPy array operations."""
    epsilon = 1e-6  # avoids division by zero when sales_count is 0
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + epsilon)

In [72]:
sales = np.array([10, 5, 0])
refunds = np.array([2, 1, 0])
print(compute_refund_ratio(sales, refunds))

[0.19999998 0.19999996 0.        ]
